### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

In [3]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_mcp_server_fqdn = ! terraform output -raw aca_mcp_server_fqdn
aca_mcp_server_fqdn = aca_mcp_server_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
MCP Server Endpoint: mcp-server.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 4096 # 8736 # 131072 # 512
)

In [25]:
response = model.stream([HumanMessage(content="What is Azure Container Apps?")])

for chunk in response:
    print(chunk.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft. It is designed to allow developers to build and deploy modern, microservices-based applications without having to manage the underlying infrastructure (like virtual machines or Kubernetes clusters).

Essentially, it is a **"Kubernetes-lite"** experience. It is built on top of Azure Kubernetes Service (AKS), KEDA, Dapr, and Envoy, but it hides all the complexity of those tools from the user.

Here is a detailed breakdown of what makes Azure Container Apps unique:

---

### 1. Core Value Proposition: "Serverless Containers"
In a traditional Kubernetes setup (like AKS), you have to manage node pools, scaling rules, and cluster upgrades. With Container Apps, you simply provide the **container image** (from Docker Hub or Azure Container Registry), and Azure handles the rest.

### 2. Key Technical Pillars
Azure Container Apps integrates several open-source technologies under the hood:

*   

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [44]:
# Invoke a specific tool
import json

result = await mcp_tools_web_search[0].ainvoke({"query": "Azure Container Apps", "limit": 3, "engines":["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "Azure Container Apps",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 3,
  "results": [
    {
      "title": "Azure Container Apps overview | Microsoft Learn",
      "url": "https://learn.microsoft.com/en-us/azure/container-apps/overview",
      "description": "<b>Azure</b> <b>Container</b> <b>Apps</b> is a serverless platform that allows you to maintain less infrastructure and save costs while running containerized applications. Instead of worrying about server configuration, <b>container</b> orchestration, and deployment details, <b>Container</b> <b>Apps</b> provides all the up-to-date server resources required to keep your applications ...",
      "source": "learn.microsoft.com",
      "engine": "duckduckgo"
    },
    {
      "title": "Azure Container Apps: Your Complete 2025 Guide to Serverless Container ...",
      "url": "https://kunaldaskd.medium.com/azure-container-apps-your-complete-2025-guide-to-serverless-container-deployment-de6ef2ef1f1a",
      "desc

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools

In [ ]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor],
    
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
# model.max_completion_tokens=131072
agent_with_mcp = create_agent(model, mcp_tools_web_search)

### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [ ]:
from langchain_core.messages import HumanMessage

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""Search the web for the latest news about Azure Container Apps in 2026.
                                          Limit search to 2 results only.
                                          Fetch and analyse each web page one by one.
                                          Use official sources only.""")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()